In [2]:
import sqlite3
import json
import re
import pickle
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import brier_score_loss
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, roc_auc_score, roc_curve
from xgboost import XGBClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

In [3]:
# Setup Directory
os.chdir("/Users/admin/Purdue + CS/MLB-AI-Betting/src/modelDevelopment")
os.makedirs("training/model_files", exist_ok=True)

# Connect to database
conn = sqlite3.connect("../../databases/MLB_Betting.db")
print("Libraries loaded and database connected.")

Libraries loaded and database connected.


In [4]:
# Convert moneyline to net profit per $1 bet
def moneyLineToPayout(odds):
    if isinstance(odds, str):
        odds = odds.strip()
        if odds.startswith('+'):
            odds = odds[1:]  # remove the '+' sign
        odds = int(odds)    

    if odds < 0:
        return 100 / -odds
    else:
        return odds / 100

# Calculate unit size and expected value
def calculateUnitSize(model_home_confidence, model_away_confidence, home_vegas_odds, away_vegas_odds):
    """
    Given model confidence and Vegas odds, compute expected value (EV) for both teams.
    Return:
    - best_team: 'home' or 'away' if positive EV, otherwise None
    - unit_size: scaled by 5 * ROI if EV > 0
    - ROI: expected ROI from betting on the better side
    """
    # Net payout per $1
    home_payout = moneyLineToPayout(home_vegas_odds)
    away_payout = moneyLineToPayout(away_vegas_odds)

    home_ev = model_home_confidence * home_payout + model_away_confidence * -1
    away_ev = model_home_confidence * -1 + model_away_confidence * away_payout

    # if both EVs are less than 0 we don't bet on it
    if (home_ev <= 0 and away_ev <= 0):
        return None, 0, 0
    
    # returns which team to bet on, the unit size, and the expected ROI
    if home_ev > away_ev:
        roi = home_ev / home_payout
        return 'home', round(roi * 5, 3), round(roi * 100, 2)
    else:
        roi = away_ev / away_payout
        return 'away', round(roi * 5, 3), round(roi * 100, 2)

print("Betting logic functions loaded.")

Betting logic functions loaded.


# RESIDUAL TESTING METHOD #

In [11]:
import sys
import sqlite3
import pandas as pd
import json
import numpy as np
import os
import pickle
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import brier_score_loss, mean_absolute_error
from xgboost import XGBRegressor
from sklearn.ensemble import HistGradientBoostingRegressor

# 1. Fetch fresh data from SQL 
# Note: Adjust the DB path depending on where your notebook is located
conn = sqlite3.connect("../../databases/MLB_Betting.db") 
join_query = """
SELECT 
    f.game_id,
    og.season,
    og.home_team AS og_home,
    og.away_team AS og_away,
    og.home_score,
    og.away_score,
    ot.home_team AS ot_home,
    ot.away_team AS ot_away,
    ot.home_close_ml,
    ot.away_close_ml,
    f.features_json
FROM Features f
INNER JOIN OldGames og ON f.game_id = og.game_id
INNER JOIN Odds_Temp ot ON f.game_id = ot.game_id
"""
df = pd.read_sql_query(join_query, conn)
conn.close()

# 2. Parse JSON Features
print("Unpacking JSON features...")
features_df = pd.json_normalize(df["features_json"].apply(json.loads))

# 3. Base Target Definition
df["label"] = (df["home_score"] > df["away_score"]).astype(int)
cols_to_drop = ['label', 'home_team_id', 'away_team_id']
features_df = features_df.drop(columns=[c for c in cols_to_drop if c in features_df.columns])

# 4. Create Difference Features (Home - Away)
print("Creating difference features dynamically...")
diff_features = pd.DataFrame()
home_cols = [col for col in features_df.columns if '_home_' in col]

for home_col in home_cols:
    away_col = home_col.replace('_home_', '_away_')
    if away_col in features_df.columns:
        diff_name = home_col.replace('_home_', '_diff_')
        diff_features[diff_name] = features_df[home_col] - features_df[away_col]

df = pd.concat([df.drop(columns=["features_json"]), diff_features], axis=1)
feature_names = diff_features.columns.tolist()

# 5. Vegas Implied Probability & RESIDUAL CALCULATION (The Quant Pivot)
print("Calculating Vegas Residuals and Time-Decay Weights...")
df["home_implied"] = df["home_close_ml"].apply(lambda x: np.nan if pd.isna(x) else (100/(x+100) if x>0 else -x/(-x+100)))
df["away_implied"] = df["away_close_ml"].apply(lambda x: np.nan if pd.isna(x) else (100/(x+100) if x>0 else -x/(-x+100)))
df["vegas_fair_prob"] = df["home_implied"] / (df["home_implied"] + df["away_implied"])
df = df.dropna(subset=["home_close_ml", "away_close_ml", "vegas_fair_prob"])

# THE NEW TARGET: How much did Vegas miss by?
df["vegas_residual"] = df["label"] - df["vegas_fair_prob"]

# FIX: Convert season to integer BEFORE doing math on it
df["season"] = df["season"].astype(int)

# THE TIME DECAY: 2023+ gets full weight, older seasons get progressively less
df["sample_weight"] = df["season"].apply(lambda x: 1.0 if x >= 2023 else max(0.2, 1.0 - (2023 - x) * 0.1))

df.drop(columns=["home_implied", "away_implied"], inplace=True)

# 6. Season Split (Train: 2021-2023, Test: 2024, Sim: 2025)
# WE ARE NUKING 2020 AND OLDER HERE
df_train = df[(df["season"] >= 2021) & (df["season"] <= 2023)]
df_test  = df[df["season"] == 2024]
df_sim   = df[df["season"] == 2025].copy()

# Notice we DO NOT append vegas_fair_prob to feature_names anymore!
X_train = df_train[feature_names]
X_test  = df_test[feature_names]

# The new target is the continuous residual error
Y_train_residual = df_train["vegas_residual"]
Y_test_residual  = df_test["vegas_residual"]

# We keep the actual labels and vegas probs aside strictly for evaluation scoring
Y_test_actual_label = df_test["label"]
vegas_probs_test = df_test["vegas_fair_prob"]
weights_train = df_train["sample_weight"]

print(f"\nTraining (2021-2023): {X_train.shape[0]} games")
print(f"Testing (2024):       {X_test.shape[0]} games")
print(f"Simulation (2025):    {df_sim.shape[0]} games")

# 7. FEATURE PRUNING (Regression Based)
print("\nPruning features based on Residual importance...")
pruning_model = XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.05, random_state=42)
pruning_model.fit(X_train, Y_train_residual, sample_weight=weights_train)

importance_df = pd.DataFrame({'feature': feature_names, 'importance': pruning_model.feature_importances_}).sort_values('importance', ascending=False)
IMPORTANCE_THRESHOLD = 0.015 # Lowered slightly since residuals distribute importance differently
features_to_keep = importance_df[importance_df['importance'] >= IMPORTANCE_THRESHOLD]['feature'].tolist()

print(f"Pruned from {len(feature_names)} → {len(features_to_keep)} features")
feature_names = features_to_keep
X_train = X_train[feature_names]
X_test  = X_test[feature_names]

# 8. SCALING
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_names)
X_test_scaled  = pd.DataFrame(scaler.transform(X_test), columns=feature_names)

os.makedirs("training/model_files", exist_ok=True)
with open("training/model_files/scaler_residual.pkl", "wb") as f:
    pickle.dump(scaler, f)
with open("training/model_files/feature_names_residual.pkl", "wb") as f:
    pickle.dump(feature_names, f)

# 9. MODEL TRAINING (Regression)
model_configs = {
    "XGBoost_Residual": XGBRegressor(
        n_estimators=150,
        max_depth=3,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='reg:squarederror',
        random_state=42
    ),
    "Gradient_Boosting_Residual": HistGradientBoostingRegressor(
        max_iter=150,
        max_depth=3,
        l2_regularization=0.1,
        random_state=42
    )
}

# Baseline check
vegas_brier = brier_score_loss(Y_test_actual_label, vegas_probs_test)
print(f"\n[BASELINE] Vegas Fair Probability Brier Score (2024): {vegas_brier:.4f}\n")

# Train, Evaluate, and SAVE
for name, model in model_configs.items():
    print(f"Training {name}...")
    
    # Fit the regressor
    if "XGBoost" in name:
        model.fit(X_train_scaled, Y_train_residual, sample_weight=weights_train)
    else:
        model.fit(X_train_scaled, Y_train_residual, sample_weight=weights_train.values)
    
    # Predict the RESIDUAL (how much Vegas is off by)
    predicted_residual = model.predict(X_test_scaled)
    
    # Reconstruct the final probability: Vegas Prob + Our Predicted Adjustment
    final_model_prob = vegas_probs_test + predicted_residual
    
    # Clip probabilities to be valid (between 1% and 99%)
    final_model_prob = np.clip(final_model_prob, 0.01, 0.99)
    
    # Score the new probabilities
    mae = mean_absolute_error(Y_test_residual, predicted_residual)
    brier = brier_score_loss(Y_test_actual_label, final_model_prob)
    diff = vegas_brier - brier
    indicator = "✅ BEAT VEGAS" if diff > 0 else "❌ WORSE"
    
    print(f"--- {name.replace('_', ' ')} ---")
    print(f"Residual MAE: {mae:.4f}")
    print(f"Brier Score:  {brier:.4f} ({indicator})")
    if diff > 0:
        print(f"Edge over House: {diff:.5f}")
        
    # Save the Regressor
    model_filename = f"training/model_files/{name.lower()}_new.pkl"
    with open(model_filename, "wb") as f:
        pickle.dump(model, f)

print("\nResidual training and saving complete.")

Unpacking JSON features...
Creating difference features dynamically...
Calculating Vegas Residuals and Time-Decay Weights...

Training (2021-2023): 7035 games
Testing (2024):       2350 games
Simulation (2025):    2350 games

Pruning features based on Residual importance...
Pruned from 36 → 36 features

[BASELINE] Vegas Fair Probability Brier Score (2024): 0.2410

Training XGBoost_Residual...
--- XGBoost Residual ---
Residual MAE: 0.4830
Brier Score:  0.2424 (❌ WORSE)
Training Gradient_Boosting_Residual...
--- Gradient Boosting Residual ---
Residual MAE: 0.4835
Brier Score:  0.2460 (❌ WORSE)

Residual training and saving complete.


# INCORPORATING PROFIT INTO THE LOSS FUNCTION #

In [12]:
import sys
import sqlite3
import pandas as pd
import json
import numpy as np
import os
import pickle
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import brier_score_loss
from xgboost import XGBClassifier

# 1. Fetch fresh data from SQL 
conn = sqlite3.connect("../../databases/MLB_Betting.db") 
join_query = """
SELECT 
    f.game_id,
    og.season,
    og.home_team AS og_home,
    og.away_team AS og_away,
    og.home_score,
    og.away_score,
    ot.home_team AS ot_home,
    ot.away_team AS ot_away,
    ot.home_close_ml,
    ot.away_close_ml,
    f.features_json
FROM Features f
INNER JOIN OldGames og ON f.game_id = og.game_id
INNER JOIN Odds_Temp ot ON f.game_id = ot.game_id
"""
df = pd.read_sql_query(join_query, conn)
conn.close()

print("Unpacking JSON features...")
features_df = pd.json_normalize(df["features_json"].apply(json.loads))

# 3. Base Target Definition
df["label"] = (df["home_score"] > df["away_score"]).astype(int)
cols_to_drop = ['label', 'home_team_id', 'away_team_id']
features_df = features_df.drop(columns=[c for c in cols_to_drop if c in features_df.columns])

# 4. Create Difference Features (Home - Away)
print("Creating difference features dynamically...")
diff_features = pd.DataFrame()
home_cols = [col for col in features_df.columns if '_home_' in col]

for home_col in home_cols:
    away_col = home_col.replace('_home_', '_away_')
    if away_col in features_df.columns:
        diff_name = home_col.replace('_home_', '_diff_')
        diff_features[diff_name] = features_df[home_col] - features_df[away_col]

df = pd.concat([df.drop(columns=["features_json"]), diff_features], axis=1)
feature_names = diff_features.columns.tolist()

# 5. THE PROFIT-WEIGHTED MATH
print("Calculating Payout Weights for Loss Function...")
# Convert Moneyline to Implied Probability (including vig)
df["home_implied"] = df["home_close_ml"].apply(lambda x: np.nan if pd.isna(x) else (100/(x+100) if x>0 else -x/(-x+100)))
df["away_implied"] = df["away_close_ml"].apply(lambda x: np.nan if pd.isna(x) else (100/(x+100) if x>0 else -x/(-x+100)))

# Calculate Decimal Odds (Payout Multiplier) = 1 / Implied Prob
df["home_decimal"] = 1.0 / df["home_implied"]
df["away_decimal"] = 1.0 / df["away_implied"]

# Clean missing odds
df = df.dropna(subset=["home_close_ml", "away_close_ml"])
df["season"] = df["season"].astype(int)

# --- THE SECRET SAUCE ---
# The weight of the game is exactly equal to the decimal payout of the winning team
df["payout_weight"] = np.where(df["label"] == 1, df["home_decimal"], df["away_decimal"])

# Time-decay: Multiply the payout weight by the era weight (modern games matter more)
df["era_weight"] = df["season"].apply(lambda x: 1.0 if x >= 2023 else max(0.2, 1.0 - (2023 - x) * 0.1))
df["final_sample_weight"] = df["payout_weight"] * df["era_weight"]

# Drop Vegas columns from features so the model doesn't get lazy
df.drop(columns=["home_implied", "away_implied", "home_decimal", "away_decimal", "payout_weight", "era_weight"], inplace=True)

# 6. Season Split (Train: 2021-2023, Test: 2024, Sim: 2025)
# WE ARE NUKING 2020 AND OLDER HERE
df_train = df[(df["season"] >= 2021) & (df["season"] <= 2023)]
df_test  = df[df["season"] == 2024]
df_sim   = df[df["season"] == 2025].copy()

X_train = df_train[feature_names]
X_test  = df_test[feature_names]
Y_train = df_train["label"]
Y_test  = df_test["label"]
weights_train = df_train["final_sample_weight"]

print(f"\nTraining (2021-2023): {X_train.shape[0]} games")
print(f"Testing (2024):       {X_test.shape[0]} games")

# 7. FEATURE PRUNING (Profit-Weighted)
print("\nPruning features based on Profit importance...")
pruning_model = XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.05, random_state=42)
pruning_model.fit(X_train, Y_train, sample_weight=weights_train)

importance_df = pd.DataFrame({'feature': feature_names, 'importance': pruning_model.feature_importances_}).sort_values('importance', ascending=False)
IMPORTANCE_THRESHOLD = 0.015 
features_to_keep = importance_df[importance_df['importance'] >= IMPORTANCE_THRESHOLD]['feature'].tolist()

print(f"Pruned from {len(feature_names)} → {len(features_to_keep)} features")
feature_names = features_to_keep
X_train = X_train[feature_names]
X_test  = X_test[feature_names]

# 8. SCALING
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_names)
X_test_scaled  = pd.DataFrame(scaler.transform(X_test), columns=feature_names)

os.makedirs("training/model_files", exist_ok=True)
with open("training/model_files/scaler_profit.pkl", "wb") as f:
    pickle.dump(scaler, f)
with open("training/model_files/feature_names_profit.pkl", "wb") as f:
    pickle.dump(feature_names, f)

# 9. MODEL TRAINING (Profit-Weighted Classifier)
print("\nTraining Profit-Weighted XGBoost Classifier...")
profit_model = XGBClassifier(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42
)

# Pass the custom payout weights directly into the loss function
profit_model.fit(X_train_scaled, Y_train, sample_weight=weights_train)

# Calculate standard Brier score just for a baseline check
y_proba = profit_model.predict_proba(X_test_scaled)[:, 1]
brier = brier_score_loss(Y_test, y_proba)
print(f"--- XGBoost Profit Weighted ---")
print(f"Raw Brier Score: {brier:.4f} (Note: Brier score matters less here, we optimized for EV)")

# Save the model
model_filename = "training/model_files/xgboost_profit_weighted.pkl"
with open(model_filename, "wb") as f:
    pickle.dump(profit_model, f)

print("\n✅ Profit-Weighted training and saving complete.")

Unpacking JSON features...
Creating difference features dynamically...
Calculating Payout Weights for Loss Function...

Training (2021-2023): 7035 games
Testing (2024):       2350 games

Pruning features based on Profit importance...
Pruned from 36 → 36 features

Training Profit-Weighted XGBoost Classifier...
--- XGBoost Profit Weighted ---
Raw Brier Score: 0.2513 (Note: Brier score matters less here, we optimized for EV)

✅ Profit-Weighted training and saving complete.
